# XRD Diffusion Model Validation Suite

This notebook provides comprehensive validation of the XRD diffusion augmentation pipeline, answering key questions about:
- Model stochasticity and controllability
- Timestep effects on augmentation quality
- DTW distance impact on transformations
- Real-world applicability and test set performance

## Table of Contents
1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Model Stochasticity Analysis](#2-model-stochasticity-analysis)
3. [Timestep Effect Investigation](#3-timestep-effect-investigation)
4. [DTW Distance Impact Study](#4-dtw-distance-impact-study)
5. [Real-world Applicability Testing](#5-real-world-applicability-testing)
6. [Comprehensive Validation Suite](#6-comprehensive-validation-suite)
7. [Interactive Exploration](#7-interactive-exploration)
8. [Summary and Conclusions](#8-summary-and-conclusions)

## 1. Setup and Data Loading

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output

# Statistical analysis
from scipy import stats
from scipy.signal import find_peaks
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Import diffusion modules
import sys
sys.path.append('diffusion')

from diffusion.datasets.xrd_dataset import XRDTransformDataset
from diffusion.models.complete_model import DiffusionAugmentor, ImprovedDiffusionDenoiser
from diffusion.diffusion.process import DiffusionProcess
from diffusion.training.config import TrainingConfig
from diffusion.visualization.plotting import plot_training_history, plot_overlay_sample

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Device configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Load the XRD dataset
print("Loading XRD dataset...")
dataset_path = "data/xrd_dataset_labeled_dtw_window.pt"
dataset_dict = torch.load(dataset_path, map_location=device)

synth_xrd = dataset_dict["synth_xrd"]
real_xrd = dataset_dict["real_xrd"] 
dtw_distances = dataset_dict["fast_dtw_distance"]

print(f"Dataset loaded:")
print(f"  Synthetic XRD shape: {synth_xrd.shape}")
print(f"  Real XRD shape: {real_xrd.shape}")
print(f"  DTW distances shape: {dtw_distances.shape}")
print(f"  DTW distance range: [{dtw_distances.min():.3f}, {dtw_distances.max():.3f}]")

# Create dataset splits
total_samples = len(synth_xrd)
train_size = int(0.7 * total_samples)
val_size = int(0.15 * total_samples)
test_size = total_samples - train_size - val_size

print(f"\nDataset splits:")
print(f"  Train: {train_size} samples")
print(f"  Validation: {val_size} samples")
print(f"  Test: {test_size} samples")

# Split the data
train_synth = synth_xrd[:train_size]
train_real = real_xrd[:train_size]
train_dtw = dtw_distances[:train_size]

val_synth = synth_xrd[train_size:train_size+val_size]
val_real = real_xrd[train_size:train_size+val_size]
val_dtw = dtw_distances[train_size:train_size+val_size]

test_synth = synth_xrd[train_size+val_size:]
test_real = real_xrd[train_size+val_size:]
test_dtw = dtw_distances[train_size+val_size:]

In [ ]:
# Load the trained model
print("Loading trained diffusion model...")

# Initialize model architecture
config = TrainingConfig()
model = DiffusionAugmentor(
    in_channels=1,
    hidden_channels=config.hidden_channels,
    time_embedding_dim=config.time_embedding_dim,
    num_res_blocks=config.num_res_blocks,
    attention_levels=config.attention_levels,
    num_levels=config.num_levels,
    temperature_condition=True
).to(device)

# Load trained weights
model_path = "diffusion/models/xrd_diffusion/best_model.pth"
try:
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Loaded model from {model_path}")
    print(f"  Trained for {checkpoint['epoch']} epochs")
    print(f"  Best validation loss: {checkpoint['val_loss']:.6f}")
except FileNotFoundError:
    print(f"⚠️  Model file not found at {model_path}")
    print("   Will use randomly initialized model for demonstration")

# Initialize diffusion process
diffusion = DiffusionProcess(
    num_timesteps=config.num_timesteps,
    schedule_type='cosine',
    device=device
)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Diffusion timesteps: {diffusion.num_timesteps}")

## 2. Model Stochasticity Analysis

First, let's investigate the model's stochastic behavior and ensure we can control deterministic vs. stochastic modes.

In [ ]:
def test_model_stochasticity(model, sample_synth, sample_dtw, n_runs=10):
    """
    Test if the model produces different outputs in stochastic mode
    and consistent outputs in deterministic mode.
    """
    model.eval()
    
    # Test sample (single pattern) - ensure correct shape [batch, channels, length]
    x = sample_synth[:1].unsqueeze(1).to(device)  # [1, 1, L]
    dtw = sample_dtw[:1].unsqueeze(1).to(device)   # [1, 1]
    t = torch.zeros(1, dtype=torch.long, device=device)  # [1]
    
    print("Testing model stochasticity...")
    print(f"Input shape: {x.shape}, DTW shape: {dtw.shape}, t shape: {t.shape}")
    
    # Test deterministic mode
    model.set_stochastic_mode(False)
    deterministic_outputs = []
    
    for i in range(n_runs):
        with torch.no_grad():
            output = model(x, t, dtw)
            deterministic_outputs.append(output.cpu().numpy())
    
    # Test stochastic mode
    model.set_stochastic_mode(True)
    stochastic_outputs = []
    
    for i in range(n_runs):
        with torch.no_grad():
            output = model(x, t, dtw)
            stochastic_outputs.append(output.cpu().numpy())
    
    # Analyze variability
    det_outputs = np.array(deterministic_outputs)
    sto_outputs = np.array(stochastic_outputs)
    
    det_std = np.std(det_outputs, axis=0).mean()
    sto_std = np.std(sto_outputs, axis=0).mean()
    
    print(f"\nDeterministic mode:")
    print(f"  Average std across runs: {det_std:.8f}")
    print(f"  Max absolute difference: {np.max(np.abs(det_outputs - det_outputs[0])):.8f}")
    
    print(f"\nStochastic mode:")
    print(f"  Average std across runs: {sto_std:.8f}")
    print(f"  Max absolute difference: {np.max(np.abs(sto_outputs - sto_outputs[0])):.8f}")
    
    print(f"\nStochasticity ratio: {sto_std / max(det_std, 1e-10):.2f}")
    
    return det_outputs, sto_outputs

# Run stochasticity test
det_outputs, sto_outputs = test_model_stochasticity(model, test_synth, test_dtw)

# Visualize the results
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot deterministic outputs
axes[0,0].plot(det_outputs[0,0,0,:], 'b-', linewidth=2, label='Run 1')
for i in range(1, min(5, len(det_outputs))):
    axes[0,0].plot(det_outputs[i,0,0,:], 'b-', alpha=0.7, linewidth=1)
axes[0,0].set_title('Deterministic Mode (Multiple Runs)')
axes[0,0].set_xlabel('Position')
axes[0,0].set_ylabel('Intensity')
axes[0,0].legend()

# Plot stochastic outputs
axes[0,1].plot(sto_outputs[0,0,0,:], 'r-', linewidth=2, label='Run 1')
for i in range(1, min(5, len(sto_outputs))):
    axes[0,1].plot(sto_outputs[i,0,0,:], 'r-', alpha=0.7, linewidth=1)
axes[0,1].set_title('Stochastic Mode (Multiple Runs)')
axes[0,1].set_xlabel('Position')
axes[0,1].set_ylabel('Intensity')
axes[0,1].legend()

# Plot variability comparison
det_std_per_pos = np.std(det_outputs, axis=0)[0,0,:]
sto_std_per_pos = np.std(sto_outputs, axis=0)[0,0,:]

axes[1,0].plot(det_std_per_pos, 'b-', label='Deterministic', linewidth=2)
axes[1,0].plot(sto_std_per_pos, 'r-', label='Stochastic', linewidth=2)
axes[1,0].set_title('Standard Deviation Across Runs')
axes[1,0].set_xlabel('Position')
axes[1,0].set_ylabel('Standard Deviation')
axes[1,0].legend()
axes[1,0].set_yscale('log')

# Plot histogram of differences
det_diffs = np.abs(det_outputs - det_outputs[0]).flatten()
sto_diffs = np.abs(sto_outputs - sto_outputs[0]).flatten()

axes[1,1].hist(det_diffs, bins=50, alpha=0.7, label='Deterministic', density=True)
axes[1,1].hist(sto_diffs, bins=50, alpha=0.7, label='Stochastic', density=True)
axes[1,1].set_title('Distribution of Absolute Differences')
axes[1,1].set_xlabel('Absolute Difference')
axes[1,1].set_ylabel('Density')
axes[1,1].legend()
axes[1,1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
def generate_stochastic_variations(model, synth_pattern, dtw_value, n_variations=10):
    """
    Generate multiple stochastic variations of the same input pattern.
    """
    model.set_stochastic_mode(True)
    model.train()  # Enable dropout and stochastic depth
    
    # Ensure correct tensor shapes
    x = synth_pattern.unsqueeze(0).unsqueeze(0).to(device)  # [1, 1, L]
    dtw = dtw_value.unsqueeze(0).to(device)    # [1, 1]
    t = torch.zeros(1, dtype=torch.long, device=device)  # Direct transformation
    
    print(f"Generating variations - Input shape: {x.shape}, DTW shape: {dtw.shape}")
    
    variations = []
    with torch.no_grad():
        for i in range(n_variations):
            output = model(x, t, dtw)
            variations.append(output[0].cpu())
    
    model.eval()
    return torch.stack(variations)

# Test with multiple samples
test_indices = [0, 10, 20]  # Different test samples
fig, axes = plt.subplots(len(test_indices), 3, figsize=(18, 6*len(test_indices)))

if len(test_indices) == 1:
    axes = axes.reshape(1, -1)

for row, idx in enumerate(test_indices):
    # Get test sample
    synth_sample = test_synth[idx]
    real_sample = test_real[idx]
    dtw_sample = test_dtw[idx]
    
    # Generate variations
    variations = generate_stochastic_variations(model, synth_sample, dtw_sample, n_variations=10)
    
    # Plot original synthetic
    axes[row, 0].plot(synth_sample.cpu(), 'k-', linewidth=2, label='Synthetic')
    axes[row, 0].set_title(f'Sample {idx}: Original Synthetic')
    axes[row, 0].set_xlabel('Position')
    axes[row, 0].set_ylabel('Intensity')
    axes[row, 0].legend()
    
    # Plot all variations
    for i, var in enumerate(variations):
        alpha = 0.7 if i == 0 else 0.4
        axes[row, 1].plot(var[0], alpha=alpha, linewidth=1)
    axes[row, 1].set_title(f'Sample {idx}: Stochastic Variations')
    axes[row, 1].set_xlabel('Position')
    axes[row, 1].set_ylabel('Intensity')
    
    # Plot comparison with real
    axes[row, 2].plot(real_sample.cpu(), 'g-', linewidth=2, label='Real', alpha=0.8)
    # Plot mean and std of variations
    var_mean = variations.mean(dim=0)[0]
    var_std = variations.std(dim=0)[0]
    axes[row, 2].plot(var_mean, 'r-', linewidth=2, label='Mean Variation')
    axes[row, 2].fill_between(range(len(var_mean)), 
                             var_mean - var_std, var_mean + var_std, 
                             alpha=0.3, color='red', label='±1 Std')
    axes[row, 2].set_title(f'Sample {idx}: Comparison with Real (DTW: {dtw_sample.item():.3f})')
    axes[row, 2].set_xlabel('Position')
    axes[row, 2].set_ylabel('Intensity')
    axes[row, 2].legend()

plt.tight_layout()
plt.show()

## 3. Timestep Effect Investigation

Now let's systematically investigate how different timesteps affect the model's behavior and augmentation strength.

In [ ]:
def analyze_timestep_effects(model, diffusion, sample_synth, sample_dtw, timesteps=None):
    """
    Analyze how different timesteps affect the model output.
    """
    if timesteps is None:
        timesteps = [0, 50, 100, 200, 400, 600, 800, 999]
    
    model.set_stochastic_mode(False)  # Deterministic for consistent comparison
    model.eval()
    
    # Ensure correct tensor shapes
    x = sample_synth.unsqueeze(0).unsqueeze(0).to(device)  # [1, 1, L]
    dtw = sample_dtw.unsqueeze(0).to(device)   # [1, 1]
    
    print(f"Analyzing timesteps - Input shape: {x.shape}, DTW shape: {dtw.shape}")
    
    results = {}
    
    with torch.no_grad():
        for t_val in tqdm(timesteps, desc="Testing timesteps"):
            t = torch.tensor([t_val], device=device, dtype=torch.long)
            
            # Get noisy version (forward diffusion)
            x_noisy, noise_true = diffusion.forward_diffusion(x, t)
            
            # Get model prediction
            noise_pred = model(x_noisy, t, dtw)
            
            # Estimate x0 from the noisy version
            alpha_bar_t = diffusion.alpha_bars[t].view(1, 1, 1)
            x0_pred = (x_noisy - torch.sqrt(1 - alpha_bar_t) * noise_pred) / torch.sqrt(alpha_bar_t)
            
            results[t_val] = {
                'noisy': x_noisy[0].cpu(),
                'noise_true': noise_true[0].cpu(),
                'noise_pred': noise_pred[0].cpu(),
                'x0_pred': x0_pred[0].cpu(),
                'noise_mse': nn.MSELoss()(noise_pred, noise_true).item(),
                'reconstruction_mse': nn.MSELoss()(x0_pred, x).item()
            }
    
    return results

# Test timestep effects on multiple samples
timestep_results = {}
test_samples = [0, 5, 10]  # Different test samples

for sample_idx in test_samples:
    print(f"\nAnalyzing timestep effects for sample {sample_idx}...")
    results = analyze_timestep_effects(
        model, diffusion, 
        test_synth[sample_idx], 
        test_dtw[sample_idx]
    )
    timestep_results[sample_idx] = results

In [ ]:
# Visualize timestep effects
def plot_timestep_analysis(timestep_results, sample_idx=0):
    results = timestep_results[sample_idx]
    timesteps = sorted(results.keys())
    
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    
    # Plot 1: Original vs noisy at different timesteps
    original = test_synth[sample_idx].cpu()
    
    for i, (ax, t) in enumerate(zip(axes[0], [0, 200, 800])):
        if t in results:
            ax.plot(original, 'k-', linewidth=2, label='Original', alpha=0.8)
            ax.plot(results[t]['noisy'][0], 'r-', linewidth=1.5, label=f'Noisy (t={t})', alpha=0.7)
            ax.set_title(f'Timestep {t}: Forward Diffusion')
            ax.set_xlabel('Position')
            ax.set_ylabel('Intensity')
            ax.legend()
    
    # Plot 2: Noise prediction quality
    for i, (ax, t) in enumerate(zip(axes[1], [0, 200, 800])):
        if t in results:
            ax.plot(results[t]['noise_true'][0], 'b-', linewidth=2, label='True Noise', alpha=0.8)
            ax.plot(results[t]['noise_pred'][0], 'orange', linewidth=1.5, label='Predicted Noise', alpha=0.7)
            ax.set_title(f'Timestep {t}: Noise Prediction (MSE: {results[t]["noise_mse"]:.4f})')
            ax.set_xlabel('Position')
            ax.set_ylabel('Noise Amplitude')
            ax.legend()
    
    # Plot 3: Reconstruction quality
    for i, (ax, t) in enumerate(zip(axes[2], [0, 200, 800])):
        if t in results:
            ax.plot(original, 'k-', linewidth=2, label='Original', alpha=0.8)
            ax.plot(results[t]['x0_pred'][0], 'g-', linewidth=1.5, label='Reconstructed', alpha=0.7)
            ax.set_title(f'Timestep {t}: Reconstruction (MSE: {results[t]["reconstruction_mse"]:.4f})')
            ax.set_xlabel('Position')
            ax.set_ylabel('Intensity')
            ax.legend()
    
    plt.suptitle(f'Timestep Analysis - Sample {sample_idx}', fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()
    
    # Plot metrics vs timestep
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    noise_mses = [results[t]['noise_mse'] for t in timesteps]
    recon_mses = [results[t]['reconstruction_mse'] for t in timesteps]
    
    axes[0].plot(timesteps, noise_mses, 'bo-', linewidth=2, markersize=6)
    axes[0].set_title('Noise Prediction Quality vs Timestep')
    axes[0].set_xlabel('Timestep')
    axes[0].set_ylabel('Noise MSE')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(timesteps, recon_mses, 'ro-', linewidth=2, markersize=6)
    axes[1].set_title('Reconstruction Quality vs Timestep')
    axes[1].set_xlabel('Timestep')
    axes[1].set_ylabel('Reconstruction MSE')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot analysis for each test sample
for sample_idx in test_samples:
    plot_timestep_analysis(timestep_results, sample_idx)

In [ ]:
# Analyze progressive augmentation behavior
def test_progressive_augmentation(diffusion, sample_pattern, timesteps=None):
    """
    Test how the diffusion process's augmentation changes with timestep.
    """
    if timesteps is None:
        timesteps = [0, 50, 100, 200, 400, 600, 800, 999]
    
    # Ensure correct tensor shape
    x = sample_pattern.unsqueeze(0).unsqueeze(0).to(device)  # [1, 1, L]
    augmented_patterns = {}
    
    print(f"Testing progressive augmentation - Input shape: {x.shape}")
    
    for t_val in timesteps:
        t = torch.tensor([t_val], device=device, dtype=torch.long)
        
        # Apply only the augmentation (without noise)
        x_aug = diffusion.augment(x, t)
        augmented_patterns[t_val] = x_aug[0].cpu()
    
    return augmented_patterns

# Test progressive augmentation
sample_idx = 0
aug_patterns = test_progressive_augmentation(diffusion, test_synth[sample_idx])

# Plot progressive augmentation
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
timesteps = sorted(aug_patterns.keys())
original = test_synth[sample_idx].cpu()

for i, t in enumerate(timesteps):
    ax = axes[i//4, i%4]
    ax.plot(original, 'k-', linewidth=2, label='Original', alpha=0.8)
    ax.plot(aug_patterns[t][0], 'r-', linewidth=1.5, label=f'Augmented (t={t})', alpha=0.7)
    ax.set_title(f'Progressive Augmentation: t={t}')
    ax.set_xlabel('Position')
    ax.set_ylabel('Intensity')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Progressive Augmentation Effects', fontsize=16)
plt.tight_layout()
plt.show()

# Quantify augmentation strength
aug_metrics = {}
for t in timesteps:
    original_np = original.numpy()
    augmented_np = aug_patterns[t][0].numpy()
    
    # Calculate various metrics
    mse = mean_squared_error(original_np, augmented_np)
    mae = mean_absolute_error(original_np, augmented_np)
    correlation = np.corrcoef(original_np, augmented_np)[0,1]
    
    # Peak analysis
    orig_peaks, _ = find_peaks(original_np, height=0.1)
    aug_peaks, _ = find_peaks(augmented_np, height=0.1)
    
    aug_metrics[t] = {
        'mse': mse,
        'mae': mae,
        'correlation': correlation,
        'peak_count_orig': len(orig_peaks),
        'peak_count_aug': len(aug_peaks),
        'peak_shift': len(aug_peaks) - len(orig_peaks)
    }

# Plot augmentation metrics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

mses = [aug_metrics[t]['mse'] for t in timesteps]
correlations = [aug_metrics[t]['correlation'] for t in timesteps]
peak_shifts = [aug_metrics[t]['peak_shift'] for t in timesteps]
peak_counts_aug = [aug_metrics[t]['peak_count_aug'] for t in timesteps]

axes[0,0].plot(timesteps, mses, 'bo-', linewidth=2, markersize=6)
axes[0,0].set_title('Augmentation Strength (MSE)')
axes[0,0].set_xlabel('Timestep')
axes[0,0].set_ylabel('MSE vs Original')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(timesteps, correlations, 'ro-', linewidth=2, markersize=6)
axes[0,1].set_title('Pattern Correlation')
axes[0,1].set_xlabel('Timestep')
axes[0,1].set_ylabel('Correlation with Original')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(timesteps, peak_shifts, 'go-', linewidth=2, markersize=6)
axes[1,0].axhline(y=0, color='k', linestyle='--', alpha=0.5)
axes[1,0].set_title('Peak Count Change')
axes[1,0].set_xlabel('Timestep')
axes[1,0].set_ylabel('Peak Count Change')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(timesteps, peak_counts_aug, 'mo-', linewidth=2, markersize=6)
axes[1,1].axhline(y=aug_metrics[0]['peak_count_orig'], color='k', linestyle='--', alpha=0.5, label='Original')
axes[1,1].set_title('Augmented Peak Count')
axes[1,1].set_xlabel('Timestep')
axes[1,1].set_ylabel('Number of Peaks')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAugmentation Analysis Summary:")
for t in [0, 200, 600, 999]:
    metrics = aug_metrics[t]
    print(f"Timestep {t:3d}: MSE={metrics['mse']:.4f}, Corr={metrics['correlation']:.3f}, Peaks={metrics['peak_count_aug']} ({metrics['peak_shift']:+d})")

## 4. DTW Distance Impact Study

Now let's investigate how DTW distance conditioning affects the model's transformations.

In [ ]:
def analyze_dtw_conditioning(model, sample_synth, dtw_values=None, fixed_timestep=0):
    """
    Analyze how different DTW conditioning values affect model output.
    """
    if dtw_values is None:
        dtw_values = np.linspace(0.0, 1.0, 11)  # 0.0 to 1.0 in steps of 0.1
    
    model.set_stochastic_mode(False)
    model.eval()
    
    x = sample_synth.unsqueeze(0).to(device)  # [1, 1, L]
    t = torch.tensor([fixed_timestep], device=device, dtype=torch.long)
    
    results = {}
    
    with torch.no_grad():
        for dtw_val in tqdm(dtw_values, desc="Testing DTW values"):
            dtw = torch.tensor([[dtw_val]], device=device, dtype=torch.float32)
            
            # Get model output
            output = model(x, t, dtw)
            
            results[dtw_val] = {
                'output': output[0].cpu(),
                'mse_vs_input': nn.MSELoss()(output, x).item()
            }
    
    return results

# Test DTW conditioning on multiple samples
dtw_results = {}
test_samples_dtw = [0, 10, 20]  # Different test samples

for sample_idx in test_samples_dtw:
    print(f"\nAnalyzing DTW conditioning for sample {sample_idx}...")
    print(f"Original DTW value: {test_dtw[sample_idx].item():.3f}")
    
    results = analyze_dtw_conditioning(model, test_synth[sample_idx])
    dtw_results[sample_idx] = results

In [ ]:
# Visualize DTW conditioning effects
def plot_dtw_analysis(dtw_results, sample_idx=0):
    results = dtw_results[sample_idx]
    dtw_values = sorted(results.keys())
    original = test_synth[sample_idx][0].cpu()
    real = test_real[sample_idx][0].cpu()
    original_dtw = test_dtw[sample_idx].item()
    
    # Plot 1: Transformation across DTW values
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Show a few key DTW values
    key_dtw_values = [0.0, 0.3, 0.6, 1.0]
    
    for i, dtw_val in enumerate(key_dtw_values[:3]):
        if dtw_val in results:
            ax = axes[0, i]
            ax.plot(original, 'k-', linewidth=2, label='Synthetic', alpha=0.8)
            ax.plot(real, 'g-', linewidth=2, label='Real', alpha=0.6)
            ax.plot(results[dtw_val]['output'][0], 'r-', linewidth=1.5, 
                   label=f'Transformed (DTW={dtw_val:.1f})', alpha=0.7)
            ax.set_title(f'DTW Conditioning: {dtw_val:.1f}')
            ax.set_xlabel('Position')
            ax.set_ylabel('Intensity')
            ax.legend()
            ax.grid(True, alpha=0.3)
    
    # Show original DTW value
    ax = axes[0, 2]
    nearest_dtw = min(dtw_values, key=lambda x: abs(x - original_dtw))
    ax.plot(original, 'k-', linewidth=2, label='Synthetic', alpha=0.8)
    ax.plot(real, 'g-', linewidth=2, label='Real', alpha=0.6)
    ax.plot(results[nearest_dtw]['output'][0], 'r-', linewidth=1.5, 
           label=f'Transformed (DTW={nearest_dtw:.1f})', alpha=0.7)
    ax.set_title(f'Original DTW: {original_dtw:.3f} (≈{nearest_dtw:.1f})')
    ax.set_xlabel('Position')
    ax.set_ylabel('Intensity')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Heatmap of transformations
    transformation_matrix = np.array([results[dtw_val]['output'][0].numpy() for dtw_val in dtw_values])
    
    im = axes[1, 0].imshow(transformation_matrix, aspect='auto', cmap='viridis')
    axes[1, 0].set_title('Transformation Heatmap')
    axes[1, 0].set_xlabel('Position')
    axes[1, 0].set_ylabel('DTW Value Index')
    axes[1, 0].set_yticks(range(0, len(dtw_values), 2))
    axes[1, 0].set_yticklabels([f'{dtw_values[i]:.1f}' for i in range(0, len(dtw_values), 2)])
    plt.colorbar(im, ax=axes[1, 0])
    
    # Plot 3: MSE vs DTW value
    mse_values = [results[dtw_val]['mse_vs_input'] for dtw_val in dtw_values]
    axes[1, 1].plot(dtw_values, mse_values, 'bo-', linewidth=2, markersize=6)
    axes[1, 1].axvline(x=original_dtw, color='r', linestyle='--', alpha=0.7, label=f'Original DTW: {original_dtw:.3f}')
    axes[1, 1].set_title('Transformation Strength vs DTW')
    axes[1, 1].set_xlabel('DTW Conditioning Value')
    axes[1, 1].set_ylabel('MSE vs Input')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Plot 4: Difference from original DTW
    if original_dtw in dtw_values or nearest_dtw in dtw_values:
        reference_output = results[nearest_dtw]['output'][0]
        differences = []
        for dtw_val in dtw_values:
            diff = np.mean(np.abs(results[dtw_val]['output'][0].numpy() - reference_output.numpy()))
            differences.append(diff)
        
        axes[1, 2].plot(dtw_values, differences, 'ro-', linewidth=2, markersize=6)
        axes[1, 2].axvline(x=original_dtw, color='k', linestyle='--', alpha=0.7, label=f'Original DTW: {original_dtw:.3f}')
        axes[1, 2].set_title(f'Difference from DTW={nearest_dtw:.1f}')
        axes[1, 2].set_xlabel('DTW Conditioning Value')
        axes[1, 2].set_ylabel('Mean Absolute Difference')
        axes[1, 2].legend()
        axes[1, 2].grid(True, alpha=0.3)
    
    plt.suptitle(f'DTW Conditioning Analysis - Sample {sample_idx}', fontsize=16, y=0.95)
    plt.tight_layout()
    plt.show()

# Plot analysis for each test sample
for sample_idx in test_samples_dtw:
    plot_dtw_analysis(dtw_results, sample_idx)

In [ ]:
# Test DTW conditioning at different timesteps
def analyze_dtw_timestep_interaction(model, sample_synth, dtw_range=(0.0, 1.0), timestep_range=(0, 500)):
    """
    Analyze how DTW conditioning interacts with different timesteps.
    """
    model.set_stochastic_mode(False)
    model.eval()
    
    dtw_values = np.linspace(dtw_range[0], dtw_range[1], 6)  # 6 DTW values
    timesteps = np.linspace(timestep_range[0], timestep_range[1], 6, dtype=int)  # 6 timesteps
    
    x = sample_synth.unsqueeze(0).to(device)  # [1, 1, L]
    
    results = np.zeros((len(dtw_values), len(timesteps)))  # MSE matrix
    
    with torch.no_grad():
        for i, dtw_val in enumerate(tqdm(dtw_values, desc="DTW values")):
            dtw = torch.tensor([[dtw_val]], device=device, dtype=torch.float32)
            
            for j, t_val in enumerate(timesteps):
                t = torch.tensor([t_val], device=device, dtype=torch.long)
                
                # Get model output
                output = model(x, t, dtw)
                mse = nn.MSELoss()(output, x).item()
                results[i, j] = mse
    
    return results, dtw_values, timesteps

# Run DTW-timestep interaction analysis
sample_idx = 0
print(f"Analyzing DTW-timestep interaction for sample {sample_idx}...")
interaction_results, dtw_vals, timestep_vals = analyze_dtw_timestep_interaction(
    model, test_synth[sample_idx]
)

# Plot interaction heatmap
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Heatmap
im = axes[0].imshow(interaction_results, aspect='auto', cmap='viridis')
axes[0].set_title('DTW-Timestep Interaction (MSE)')
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('DTW Value')
axes[0].set_xticks(range(len(timestep_vals)))
axes[0].set_xticklabels([f'{t}' for t in timestep_vals])
axes[0].set_yticks(range(len(dtw_vals)))
axes[0].set_yticklabels([f'{d:.1f}' for d in dtw_vals])
plt.colorbar(im, ax=axes[0])

# Line plots
for i, dtw_val in enumerate(dtw_vals):
    axes[1].plot(timestep_vals, interaction_results[i], 'o-', label=f'DTW={dtw_val:.1f}', alpha=0.7)

axes[1].set_title('MSE vs Timestep for Different DTW Values')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('MSE vs Input')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInteraction Analysis Summary:")
print(f"Max MSE: {interaction_results.max():.4f} at DTW={dtw_vals[np.unravel_index(interaction_results.argmax(), interaction_results.shape)[0]]:.1f}, t={timestep_vals[np.unravel_index(interaction_results.argmax(), interaction_results.shape)[1]]}")
print(f"Min MSE: {interaction_results.min():.4f} at DTW={dtw_vals[np.unravel_index(interaction_results.argmin(), interaction_results.shape)[0]]:.1f}, t={timestep_vals[np.unravel_index(interaction_results.argmin(), interaction_results.shape)[1]]}")

## 5. Real-world Applicability Testing

Now let's evaluate how well the model performs on the test set and whether the augmentations are realistic.

In [ ]:
def evaluate_test_set_performance(model, diffusion, test_synth, test_real, test_dtw, batch_size=32):
    """
    Comprehensive evaluation on the test set.
    """
    model.eval()
    model.set_stochastic_mode(False)  # Deterministic evaluation
    
    n_samples = len(test_synth)
    n_batches = (n_samples + batch_size - 1) // batch_size
    
    results = {
        'diffusion_losses': [],
        'reconstruction_losses': [],
        'direct_transform_losses': [],
        'real_similarities': [],
        'peak_correlations': []
    }
    
    loss_fn = nn.MSELoss(reduction='none')
    
    with torch.no_grad():
        for batch_idx in tqdm(range(n_batches), desc="Evaluating test set"):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, n_samples)
            
            # Get batch
            synth_batch = test_synth[start_idx:end_idx].unsqueeze(1).to(device)
            real_batch = test_real[start_idx:end_idx].unsqueeze(1).to(device)
            dtw_batch = test_dtw[start_idx:end_idx].unsqueeze(1).to(device)
            
            current_batch_size = synth_batch.shape[0]
            
            # 1. Diffusion loss (random timesteps)
            t = torch.randint(0, diffusion.num_timesteps, (current_batch_size,), device=device)
            x_t, noise = diffusion.forward_diffusion(synth_batch, t)
            noise_pred = model(x_t, t, dtw_batch)
            diff_loss = loss_fn(noise_pred, noise).mean(dim=[1, 2])
            results['diffusion_losses'].extend(diff_loss.cpu().numpy())
            
            # 2. Direct transformation (t=0)
            t_zero = torch.zeros(current_batch_size, dtype=torch.long, device=device)
            transformed = model(synth_batch, t_zero, dtw_batch)
            transform_loss = loss_fn(transformed, real_batch).mean(dim=[1, 2])
            results['direct_transform_losses'].extend(transform_loss.cpu().numpy())
            
            # 3. Reconstruction test (add noise, then denoise)
            t_recon = torch.full((current_batch_size,), 100, device=device)  # Moderate noise
            x_noisy, noise_true = diffusion.forward_diffusion(synth_batch, t_recon)
            noise_pred_recon = model(x_noisy, t_recon, dtw_batch)
            
            # Estimate x0
            alpha_bar_t = diffusion.alpha_bars[t_recon].view(-1, 1, 1)
            x0_pred = (x_noisy - torch.sqrt(1 - alpha_bar_t) * noise_pred_recon) / torch.sqrt(alpha_bar_t)
            recon_loss = loss_fn(x0_pred, synth_batch).mean(dim=[1, 2])
            results['reconstruction_losses'].extend(recon_loss.cpu().numpy())
            
            # 4. Similarity to real patterns
            for i in range(current_batch_size):
                synth_np = synth_batch[i, 0].cpu().numpy()
                real_np = real_batch[i, 0].cpu().numpy()
                transformed_np = transformed[i, 0].cpu().numpy()
                
                # Correlations
                real_similarity = np.corrcoef(transformed_np, real_np)[0, 1]
                results['real_similarities'].append(real_similarity)
                
                # Peak analysis
                try:
                    synth_peaks, _ = find_peaks(synth_np, height=0.1)
                    real_peaks, _ = find_peaks(real_np, height=0.1)
                    
                    if len(synth_peaks) > 0 and len(real_peaks) > 0:
                        # Simple peak position correlation
                        peak_corr = len(set(synth_peaks) & set(real_peaks)) / max(len(synth_peaks), len(real_peaks))
                        results['peak_correlations'].append(peak_corr)
                except:
                    results['peak_correlations'].append(0.0)
    
    # Convert to numpy arrays
    for key in results:
        results[key] = np.array(results[key])
    
    return results

# Evaluate on test set
print("Evaluating model performance on test set...")
test_results = evaluate_test_set_performance(model, diffusion, test_synth, test_real, test_dtw)

In [ ]:
# Analyze and visualize test results
def plot_test_performance(test_results):
    """
    Plot comprehensive test performance analysis.
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Loss distributions
    axes[0, 0].hist(test_results['diffusion_losses'], bins=50, alpha=0.7, label='Diffusion Loss')
    axes[0, 0].hist(test_results['reconstruction_losses'], bins=50, alpha=0.7, label='Reconstruction Loss')
    axes[0, 0].hist(test_results['direct_transform_losses'], bins=50, alpha=0.7, label='Transform Loss')
    axes[0, 0].set_title('Loss Distributions')
    axes[0, 0].set_xlabel('Loss Value')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].legend()
    axes[0, 0].set_yscale('log')
    
    # 2. Real similarity distribution
    axes[0, 1].hist(test_results['real_similarities'], bins=50, alpha=0.7, color='green')
    axes[0, 1].axvline(np.mean(test_results['real_similarities']), color='red', 
                      linestyle='--', label=f'Mean: {np.mean(test_results["real_similarities"]):.3f}')
    axes[0, 1].set_title('Similarity to Real Patterns')
    axes[0, 1].set_xlabel('Correlation with Real')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].legend()
    
    # 3. Peak correlation distribution
    axes[0, 2].hist(test_results['peak_correlations'], bins=30, alpha=0.7, color='orange')
    axes[0, 2].axvline(np.mean(test_results['peak_correlations']), color='red', 
                      linestyle='--', label=f'Mean: {np.mean(test_results["peak_correlations"]):.3f}')
    axes[0, 2].set_title('Peak Position Correlation')
    axes[0, 2].set_xlabel('Peak Correlation')
    axes[0, 2].set_ylabel('Frequency')
    axes[0, 2].legend()
    
    # 4. Loss vs DTW relationship
    # Sort by DTW values for plotting
    dtw_sorted_idx = np.argsort(test_dtw.cpu().numpy())
    sorted_dtw = test_dtw.cpu().numpy()[dtw_sorted_idx]
    sorted_transform_loss = test_results['direct_transform_losses'][dtw_sorted_idx]
    
    # Bin and average for cleaner plot
    n_bins = 20
    bin_edges = np.linspace(sorted_dtw.min(), sorted_dtw.max(), n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    binned_losses = []
    
    for i in range(n_bins):
        mask = (sorted_dtw >= bin_edges[i]) & (sorted_dtw < bin_edges[i + 1])
        if mask.sum() > 0:
            binned_losses.append(sorted_transform_loss[mask].mean())
        else:
            binned_losses.append(np.nan)
    
    axes[1, 0].plot(bin_centers, binned_losses, 'bo-', markersize=6)
    axes[1, 0].set_title('Transform Loss vs DTW Distance')
    axes[1, 0].set_xlabel('DTW Distance')
    axes[1, 0].set_ylabel('Average Transform Loss')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Similarity vs DTW relationship  
    sorted_similarity = test_results['real_similarities'][dtw_sorted_idx]
    binned_similarities = []
    
    for i in range(n_bins):
        mask = (sorted_dtw >= bin_edges[i]) & (sorted_dtw < bin_edges[i + 1])
        if mask.sum() > 0:
            binned_similarities.append(sorted_similarity[mask].mean())
        else:
            binned_similarities.append(np.nan)
    
    axes[1, 1].plot(bin_centers, binned_similarities, 'go-', markersize=6)
    axes[1, 1].set_title('Real Similarity vs DTW Distance')
    axes[1, 1].set_xlabel('DTW Distance')
    axes[1, 1].set_ylabel('Average Similarity to Real')
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Performance summary statistics
    stats_text = f"""
Test Set Performance Summary:

Diffusion Loss: {np.mean(test_results['diffusion_losses']):.4f} ± {np.std(test_results['diffusion_losses']):.4f}
Reconstruction Loss: {np.mean(test_results['reconstruction_losses']):.4f} ± {np.std(test_results['reconstruction_losses']):.4f}
Transform Loss: {np.mean(test_results['direct_transform_losses']):.4f} ± {np.std(test_results['direct_transform_losses']):.4f}

Real Similarity: {np.mean(test_results['real_similarities']):.3f} ± {np.std(test_results['real_similarities']):.3f}
Peak Correlation: {np.mean(test_results['peak_correlations']):.3f} ± {np.std(test_results['peak_correlations']):.3f}

Samples with high similarity (>0.8): {np.sum(test_results['real_similarities'] > 0.8) / len(test_results['real_similarities']) * 100:.1f}%
Samples with low loss (<0.01): {np.sum(test_results['direct_transform_losses'] < 0.01) / len(test_results['direct_transform_losses']) * 100:.1f}%
    """.strip()
    
    axes[1, 2].text(0.05, 0.95, stats_text, transform=axes[1, 2].transAxes, 
                   fontsize=10, verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    axes[1, 2].set_xlim(0, 1)
    axes[1, 2].set_ylim(0, 1)
    axes[1, 2].axis('off')
    axes[1, 2].set_title('Performance Summary')
    
    plt.tight_layout()
    plt.show()

# Plot test performance
plot_test_performance(test_results)

# Print detailed statistics
print("\n" + "="*60)
print("DETAILED TEST SET PERFORMANCE ANALYSIS")
print("="*60)

print(f"\nTotal test samples: {len(test_results['diffusion_losses'])}")

print(f"\nLoss Statistics:")
for loss_name, losses in [("Diffusion", test_results['diffusion_losses']),
                          ("Reconstruction", test_results['reconstruction_losses']),
                          ("Direct Transform", test_results['direct_transform_losses'])]:
    print(f"  {loss_name:15s}: Mean={np.mean(losses):.6f}, Std={np.std(losses):.6f}, "
          f"Median={np.median(losses):.6f}, Min={np.min(losses):.6f}, Max={np.max(losses):.6f}")

print(f"\nSimilarity Statistics:")
similarities = test_results['real_similarities']
print(f"  Real Similarity   : Mean={np.mean(similarities):.4f}, Std={np.std(similarities):.4f}, "
      f"Median={np.median(similarities):.4f}")

peak_corrs = test_results['peak_correlations']
print(f"  Peak Correlation  : Mean={np.mean(peak_corrs):.4f}, Std={np.std(peak_corrs):.4f}, "
      f"Median={np.median(peak_corrs):.4f}")

print(f"\nQuality Metrics:")
print(f"  High similarity samples (>0.8): {np.sum(similarities > 0.8)} ({np.sum(similarities > 0.8)/len(similarities)*100:.1f}%)")
print(f"  Good similarity samples (>0.6): {np.sum(similarities > 0.6)} ({np.sum(similarities > 0.6)/len(similarities)*100:.1f}%)")
print(f"  Low transform loss (<0.01): {np.sum(test_results['direct_transform_losses'] < 0.01)} ({np.sum(test_results['direct_transform_losses'] < 0.01)/len(test_results['direct_transform_losses'])*100:.1f}%)")
print(f"  Good peak correlation (>0.5): {np.sum(peak_corrs > 0.5)} ({np.sum(peak_corrs > 0.5)/len(peak_corrs)*100:.1f}%)")

## 6. Comprehensive Validation Suite

Let's run a comprehensive validation that combines all our analyses.

In [ ]:
def comprehensive_validation_suite(model, diffusion, test_synth, test_real, test_dtw):
    """
    Run a comprehensive validation suite covering all aspects.
    """
    print("Running Comprehensive Validation Suite...")
    print("="*60)
    
    validation_results = {}
    
    # 1. Stochasticity Validation
    print("\n1. Testing Model Stochasticity...")
    det_outputs, sto_outputs = test_model_stochasticity(model, test_synth[:5], test_dtw[:5], n_runs=5)
    
    det_std = np.std(det_outputs, axis=0).mean()
    sto_std = np.std(sto_outputs, axis=0).mean()
    stochasticity_ratio = sto_std / max(det_std, 1e-10)
    
    validation_results['stochasticity'] = {
        'deterministic_std': det_std,
        'stochastic_std': sto_std,
        'ratio': stochasticity_ratio,
        'pass': stochasticity_ratio > 10  # Stochastic should be much more variable
    }
    
    print(f"   ✓ Deterministic std: {det_std:.8f}")
    print(f"   ✓ Stochastic std: {sto_std:.8f}")
    print(f"   ✓ Ratio: {stochasticity_ratio:.2f} {'(PASS)' if validation_results['stochasticity']['pass'] else '(FAIL)'}")
    
    # 2. Timestep Effect Validation
    print("\n2. Testing Timestep Effects...")
    sample_idx = 0
    timestep_results = analyze_timestep_effects(model, diffusion, test_synth[sample_idx], test_dtw[sample_idx])
    
    timesteps = sorted(timestep_results.keys())
    noise_mses = [timestep_results[t]['noise_mse'] for t in timesteps]
    recon_mses = [timestep_results[t]['reconstruction_mse'] for t in timesteps]
    
    # Validate that higher timesteps generally have higher reconstruction error
    timestep_correlation = stats.spearmanr(timesteps, recon_mses)[0]
    
    validation_results['timestep_effects'] = {
        'correlation_with_recon_error': timestep_correlation,
        'pass': timestep_correlation > 0.5  # Should be positive correlation
    }
    
    print(f"   ✓ Timestep-reconstruction error correlation: {timestep_correlation:.3f} {'(PASS)' if validation_results['timestep_effects']['pass'] else '(FAIL)'}")
    
    # 3. DTW Conditioning Validation
    print("\n3. Testing DTW Conditioning...")
    dtw_results = analyze_dtw_conditioning(model, test_synth[sample_idx])
    
    dtw_values = sorted(dtw_results.keys())
    transform_mses = [dtw_results[d]['mse_vs_input'] for d in dtw_values]
    
    # Check if DTW conditioning affects output (should have variation)
    dtw_effect_std = np.std(transform_mses)
    dtw_effect_range = max(transform_mses) - min(transform_mses)
    
    validation_results['dtw_conditioning'] = {
        'effect_std': dtw_effect_std,
        'effect_range': dtw_effect_range,
        'pass': dtw_effect_std > 1e-4  # Should have noticeable effect
    }
    
    print(f"   ✓ DTW conditioning effect std: {dtw_effect_std:.6f} {'(PASS)' if validation_results['dtw_conditioning']['pass'] else '(FAIL)'}")
    print(f"   ✓ DTW conditioning range: {dtw_effect_range:.6f}")
    
    # 4. Real-world Similarity Validation
    print("\n4. Testing Real-world Similarity...")
    # Quick test on subset
    subset_size = min(50, len(test_synth))
    subset_results = evaluate_test_set_performance(
        model, diffusion, 
        test_synth[:subset_size], 
        test_real[:subset_size], 
        test_dtw[:subset_size], 
        batch_size=16
    )
    
    mean_similarity = np.mean(subset_results['real_similarities'])
    high_similarity_fraction = np.sum(subset_results['real_similarities'] > 0.6) / len(subset_results['real_similarities'])
    
    validation_results['real_world_similarity'] = {
        'mean_similarity': mean_similarity,
        'high_similarity_fraction': high_similarity_fraction,
        'pass': mean_similarity > 0.3 and high_similarity_fraction > 0.2  # Reasonable thresholds
    }
    
    print(f"   ✓ Mean similarity to real: {mean_similarity:.3f} {'(PASS)' if mean_similarity > 0.3 else '(FAIL)'}")
    print(f"   ✓ High similarity fraction: {high_similarity_fraction:.3f} {'(PASS)' if high_similarity_fraction > 0.2 else '(FAIL)'}")
    
    # 5. Model Stability Validation
    print("\n5. Testing Model Stability...")
    # Test with extreme inputs
    model.eval()
    with torch.no_grad():
        # Zero input
        zero_input = torch.zeros(1, 1, test_synth.shape[-1]).to(device)
        dtw_test = torch.tensor([[0.5]], device=device)
        t_test = torch.tensor([0], device=device)
        
        zero_output = model(zero_input, t_test, dtw_test)
        
        # Random input
        random_input = torch.randn(1, 1, test_synth.shape[-1]).to(device)
        random_output = model(random_input, t_test, dtw_test)
        
        # Check for NaN or inf
        stability_pass = (torch.isfinite(zero_output).all() and 
                         torch.isfinite(random_output).all() and
                         not torch.isnan(zero_output).any() and
                         not torch.isnan(random_output).any())
    
    validation_results['stability'] = {
        'pass': stability_pass
    }
    
    print(f"   ✓ Model stability: {'PASS' if stability_pass else 'FAIL'}")
    
    # Summary
    print("\n" + "="*60)
    print("VALIDATION SUMMARY")
    print("="*60)
    
    total_tests = len(validation_results)
    passed_tests = sum(1 for test in validation_results.values() if test['pass'])
    
    print(f"Tests passed: {passed_tests}/{total_tests} ({passed_tests/total_tests*100:.1f}%)")
    
    for test_name, result in validation_results.items():
        status = "✓ PASS" if result['pass'] else "✗ FAIL"
        print(f"  {test_name:20s}: {status}")
    
    overall_pass = passed_tests == total_tests
    print(f"\nOVERALL RESULT: {'✓ ALL TESTS PASSED' if overall_pass else '✗ SOME TESTS FAILED'}")
    
    return validation_results

# Run comprehensive validation
validation_results = comprehensive_validation_suite(model, diffusion, test_synth, test_real, test_dtw)

## 7. Interactive Exploration

Let's create interactive widgets to explore the model behavior in real-time.

In [ ]:
# Interactive exploration widgets
def create_interactive_explorer():
    """
    Create interactive widgets for exploring model behavior.
    """
    
    # Sample selection
    sample_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(test_synth) - 1,
        step=1,
        description='Sample:',
        continuous_update=False
    )
    
    # Timestep selection
    timestep_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=diffusion.num_timesteps - 1,
        step=10,
        description='Timestep:',
        continuous_update=False
    )
    
    # DTW conditioning
    dtw_slider = widgets.FloatSlider(
        value=0.5,
        min=0.0,
        max=1.0,
        step=0.05,
        description='DTW Value:',
        continuous_update=False
    )
    
    # Stochastic mode
    stochastic_checkbox = widgets.Checkbox(
        value=False,
        description='Enable Stochastic Mode'
    )
    
    # Number of variations
    n_variations_slider = widgets.IntSlider(
        value=5,
        min=1,
        max=20,
        step=1,
        description='Variations:',
        continuous_update=False
    )
    
    # Output widget
    output = widgets.Output()
    
    def update_plot(sample_idx, timestep, dtw_value, use_stochastic, n_variations):
        with output:
            clear_output(wait=True)
            
            # Get sample data
            synth_sample = test_synth[sample_idx]
            real_sample = test_real[sample_idx]
            original_dtw = test_dtw[sample_idx].item()
            
            # Set model mode
            model.set_stochastic_mode(use_stochastic)
            if use_stochastic:
                model.train()
            else:
                model.eval()
            
            # Prepare inputs
            x = synth_sample.unsqueeze(0).to(device)
            t = torch.tensor([timestep], device=device, dtype=torch.long)
            dtw = torch.tensor([[dtw_value]], device=device, dtype=torch.float32)
            
            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
            
            # Plot 1: Original data
            axes[0, 0].plot(synth_sample[0].cpu(), 'k-', linewidth=2, label='Synthetic', alpha=0.8)
            axes[0, 0].plot(real_sample[0].cpu(), 'g-', linewidth=2, label='Real', alpha=0.6)
            axes[0, 0].set_title(f'Sample {sample_idx} (Original DTW: {original_dtw:.3f})')
            axes[0, 0].set_xlabel('Position')
            axes[0, 0].set_ylabel('Intensity')
            axes[0, 0].legend()
            axes[0, 0].grid(True, alpha=0.3)
            
            # Plot 2: Model outputs
            with torch.no_grad():
                if use_stochastic and n_variations > 1:
                    # Multiple stochastic variations
                    for i in range(n_variations):
                        output_sample = model(x, t, dtw)
                        alpha = 0.8 if i == 0 else 0.4
                        label = 'Model Output' if i == 0 else None
                        axes[0, 1].plot(output_sample[0, 0].cpu(), 'r-', alpha=alpha, linewidth=1.5, label=label)
                else:
                    # Single output
                    output_sample = model(x, t, dtw)
                    axes[0, 1].plot(output_sample[0, 0].cpu(), 'r-', linewidth=2, label='Model Output')
                
                axes[0, 1].plot(real_sample[0].cpu(), 'g-', linewidth=2, label='Real Target', alpha=0.6)
            
            mode_str = "Stochastic" if use_stochastic else "Deterministic"
            axes[0, 1].set_title(f'{mode_str} Output (t={timestep}, DTW={dtw_value:.2f})')
            axes[0, 1].set_xlabel('Position')
            axes[0, 1].set_ylabel('Intensity')
            axes[0, 1].legend()
            axes[0, 1].grid(True, alpha=0.3)
            
            # Plot 3: Diffusion process visualization (if timestep > 0)
            if timestep > 0:
                with torch.no_grad():
                    x_noisy, noise = diffusion.forward_diffusion(x, t)
                    noise_pred = model(x_noisy, t, dtw)
                    
                    axes[1, 0].plot(synth_sample[0].cpu(), 'k-', linewidth=2, label='Original', alpha=0.8)
                    axes[1, 0].plot(x_noisy[0, 0].cpu(), 'orange', linewidth=1.5, label='Noisy', alpha=0.7)
                    
                    # Reconstruct x0
                    alpha_bar_t = diffusion.alpha_bars[t].view(1, 1, 1)
                    x0_pred = (x_noisy - torch.sqrt(1 - alpha_bar_t) * noise_pred) / torch.sqrt(alpha_bar_t)
                    axes[1, 0].plot(x0_pred[0, 0].cpu(), 'r--', linewidth=1.5, label='Reconstructed', alpha=0.7)
                    
                axes[1, 0].set_title(f'Diffusion Process (t={timestep})')
                axes[1, 0].set_xlabel('Position')
                axes[1, 0].set_ylabel('Intensity')
                axes[1, 0].legend()
                axes[1, 0].grid(True, alpha=0.3)
            else:
                axes[1, 0].text(0.5, 0.5, 'No diffusion at t=0\n(Direct transformation)', 
                               ha='center', va='center', transform=axes[1, 0].transAxes,
                               fontsize=12, bbox=dict(boxstyle='round', facecolor='lightgray'))
                axes[1, 0].set_title('Diffusion Process')
            
            # Plot 4: Parameter info
            info_text = f"""
Current Settings:
Sample Index: {sample_idx}
Timestep: {timestep}
DTW Value: {dtw_value:.3f}
Original DTW: {original_dtw:.3f}
Mode: {mode_str}
Variations: {n_variations if use_stochastic else 1}

Sample Info:
Pattern Length: {len(synth_sample[0])}
Synth Range: [{synth_sample[0].min():.3f}, {synth_sample[0].max():.3f}]
Real Range: [{real_sample[0].min():.3f}, {real_sample[0].max():.3f}]
            """.strip()
            
            axes[1, 1].text(0.05, 0.95, info_text, transform=axes[1, 1].transAxes,
                           fontsize=10, verticalalignment='top', fontfamily='monospace',
                           bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
            axes[1, 1].set_xlim(0, 1)
            axes[1, 1].set_ylim(0, 1)
            axes[1, 1].axis('off')
            axes[1, 1].set_title('Parameters & Info')
            
            plt.tight_layout()
            plt.show()
    
    # Create interactive widget
    interactive_widget = widgets.interactive(
        update_plot,
        sample_idx=sample_slider,
        timestep=timestep_slider,
        dtw_value=dtw_slider,
        use_stochastic=stochastic_checkbox,
        n_variations=n_variations_slider
    )
    
    # Layout
    controls = widgets.VBox([
        widgets.HBox([sample_slider, timestep_slider]),
        widgets.HBox([dtw_slider, stochastic_checkbox]),
        n_variations_slider
    ])
    
    full_widget = widgets.VBox([controls, output])
    
    # Initial plot
    update_plot(0, 0, 0.5, False, 5)
    
    return full_widget

# Create and display interactive explorer
print("Creating Interactive Model Explorer...")
print("Use the sliders and checkboxes below to explore model behavior:")
explorer = create_interactive_explorer()
display(explorer)

## 8. Summary and Conclusions

Let's summarize our findings and answer the original questions.

In [ ]:
# Generate comprehensive summary report
def generate_summary_report(validation_results, test_results):
    """
    Generate a comprehensive summary report answering the key questions.
    """
    
    print("\n" + "="*80)
    print("XRD DIFFUSION MODEL VALIDATION SUMMARY REPORT")
    print("="*80)
    
    print("\n🎯 KEY QUESTIONS ANSWERED:")
    print("-" * 50)
    
    # Question 1: Model Stochasticity
    print("\n1. IS THE MODEL STOCHASTIC?")
    if validation_results['stochasticity']['pass']:
        print("   ✅ YES - The model exhibits clear stochastic behavior")
        print(f"   📊 Stochastic mode is {validation_results['stochasticity']['ratio']:.1f}x more variable than deterministic")
        print("   🔧 Controllable via model.set_stochastic_mode() and model.train()/eval()")
        print("   💡 Stochastic features: dropout, stochastic depth, and variational components")
    else:
        print("   ⚠️  UNCLEAR - Stochastic behavior may be limited")
        print(f"   📊 Stochasticity ratio: {validation_results['stochasticity']['ratio']:.2f}")
    
    # Question 2: Timestep Effects
    print("\n2. HOW DOES CHANGING TIMESTEP AFFECT THE MODEL?")
    if validation_results['timestep_effects']['pass']:
        print("   ✅ PROGRESSIVE EFFECTS - Timesteps systematically control augmentation strength")
        print(f"   📈 Correlation with reconstruction error: {validation_results['timestep_effects']['correlation_with_recon_error']:.3f}")
        print("   🔄 Higher timesteps → More noise → Stronger augmentation")
        print("   🧬 Includes physics-based peak broadening (Scherrer equation)")
        print("   🎛️  t=0: Direct transformation, t>0: Diffusion with increasing noise")
    else:
        print("   ⚠️  WEAK CORRELATION - Timestep effects may be inconsistent")
    
    # Question 3: DTW Distance Impact
    print("\n3. HOW DOES DTW DISTANCE CHANGE THE MODEL?")
    if validation_results['dtw_conditioning']['pass']:
        print("   ✅ SIGNIFICANT CONDITIONING - DTW distance effectively controls transformations")
        print(f"   📊 Effect standard deviation: {validation_results['dtw_conditioning']['effect_std']:.6f}")
        print(f"   📏 Effect range: {validation_results['dtw_conditioning']['effect_range']:.6f}")
        print("   🎯 DTW values guide synthetic-to-real transformation strength")
        print("   📐 Range: 0.0 (minimal transform) to 1.0 (maximal transform)")
    else:
        print("   ⚠️  LIMITED EFFECT - DTW conditioning may not be working properly")
    
    # Question 4: Real-world Applicability
    print("\n4. IS THIS AUGMENTATION SIMILAR TO REAL-WORLD DATA?")
    if validation_results['real_world_similarity']['pass']:
        print("   ✅ REALISTIC AUGMENTATION - Good similarity to real experimental data")
        print(f"   📊 Mean similarity to real patterns: {validation_results['real_world_similarity']['mean_similarity']:.3f}")
        print(f"   🎯 High similarity samples (>0.6): {validation_results['real_world_similarity']['high_similarity_fraction']*100:.1f}%")
        print("   🧪 Model learned realistic experimental transformations")
    else:
        print("   ⚠️  POOR SIMILARITY - Augmentations may not reflect real-world data")
        print(f"   📊 Mean similarity: {validation_results['real_world_similarity']['mean_similarity']:.3f}")
    
    # Question 5: Test Set Performance
    print("\n5. DOES IT WORK ON THE TEST SET?")
    mean_sim = np.mean(test_results['real_similarities'])
    high_sim_frac = np.sum(test_results['real_similarities'] > 0.6) / len(test_results['real_similarities'])
    mean_transform_loss = np.mean(test_results['direct_transform_losses'])
    
    if mean_sim > 0.4 and high_sim_frac > 0.3:
        print("   ✅ GOOD GENERALIZATION - Model performs well on unseen test data")
    else:
        print("   ⚠️  LIMITED GENERALIZATION - Performance on test set needs improvement")
    
    print(f"   📊 Test set similarity: {mean_sim:.3f} ± {np.std(test_results['real_similarities']):.3f}")
    print(f"   🎯 Good similarity samples: {high_sim_frac*100:.1f}%")
    print(f"   💔 Transform loss: {mean_transform_loss:.4f} ± {np.std(test_results['direct_transform_losses']):.4f}")
    print(f"   📈 Peak correlation: {np.mean(test_results['peak_correlations']):.3f} ± {np.std(test_results['peak_correlations']):.3f}")
    
    # Technical Summary
    print("\n\n🔧 TECHNICAL SUMMARY:")
    print("-" * 50)
    print("Architecture: U-Net style diffusion model with stochastic components")
    print("Conditioning: DTW distance values for real-world similarity guidance")
    print("Physics: Scherrer equation for realistic peak broadening")
    print("Stochasticity: Controlled via dropout, stochastic depth, and training mode")
    print("Timesteps: Progressive augmentation from t=0 (clean) to t=999 (noisy)")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,} trainable parameters")
    print(f"Test samples: {len(test_results['real_similarities'])} samples evaluated")
    
    # Recommendations
    print("\n\n💡 RECOMMENDATIONS:")
    print("-" * 50)
    
    if validation_results['stochasticity']['pass']:
        print("✅ Stochastic augmentation is working - use model.train() for data augmentation")
    else:
        print("🔧 Consider increasing dropout rates or stochastic depth probability")
    
    if validation_results['timestep_effects']['pass']:
        print("✅ Use varying timesteps (0-500) for different augmentation strengths")
    else:
        print("🔧 Review timestep schedule and diffusion process parameters")
    
    if validation_results['dtw_conditioning']['pass']:
        print("✅ DTW conditioning is effective - use original DTW values for best results")
    else:
        print("🔧 Consider adjusting DTW conditioning architecture or normalization")
    
    if validation_results['real_world_similarity']['pass']:
        print("✅ Model produces realistic augmentations suitable for training")
        print("📈 Consider using augmented data for downstream classification tasks")
    else:
        print("🔧 May need additional training or architecture improvements")
        print("🧪 Validate with domain experts on augmentation realism")
    
    # Validation Summary
    total_tests = len(validation_results)
    passed_tests = sum(1 for test in validation_results.values() if test['pass'])
    
    print("\n\n📋 VALIDATION SUMMARY:")
    print("-" * 50)
    print(f"Overall validation: {passed_tests}/{total_tests} tests passed ({passed_tests/total_tests*100:.1f}%)")
    
    for test_name, result in validation_results.items():
        status = "✅ PASS" if result['pass'] else "❌ FAIL"
        print(f"  {test_name.replace('_', ' ').title():20s}: {status}")
    
    if passed_tests == total_tests:
        print("\n🎉 ALL VALIDATIONS PASSED - Model is ready for production use!")
    else:
        print(f"\n⚠️  {total_tests - passed_tests} validations failed - Review recommendations above")
    
    print("\n" + "="*80)

# Generate the final report
generate_summary_report(validation_results, test_results)

In [ ]:
# Save validation results for future reference
import pickle
from datetime import datetime

# Create comprehensive results dictionary
complete_results = {
    'timestamp': datetime.now().isoformat(),
    'model_info': {
        'parameters': sum(p.numel() for p in model.parameters()),
        'device': device,
        'diffusion_timesteps': diffusion.num_timesteps
    },
    'validation_results': validation_results,
    'test_results': test_results,
    'dataset_info': {
        'total_samples': len(synth_xrd),
        'train_samples': len(train_synth),
        'val_samples': len(val_synth),
        'test_samples': len(test_synth),
        'pattern_length': synth_xrd.shape[-1],
        'dtw_range': [float(dtw_distances.min()), float(dtw_distances.max())]
    }
}

# Save results
results_path = 'xrd_diffusion_validation_results.pkl'
with open(results_path, 'wb') as f:
    pickle.dump(complete_results, f)

print(f"\n💾 Validation results saved to: {results_path}")
print("\n📝 This notebook has provided comprehensive validation of your XRD diffusion model.")
print("   Use the interactive explorer above to continue investigating model behavior.")
print("   Refer to the summary report for detailed answers to your original questions.")